In [1]:
!python --version

Python 3.11.9


In [2]:
DATASET_DIR = "../../datasets/"

import pathlib
OUTPUT_DIR = pathlib.Path(DATASET_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_BRAND_RXNORM = "brand_rxnorm.csv"
RXNORM_BRAND_PATH = OUTPUT_DIR.joinpath(FILE_BRAND_RXNORM)
print(f"RxNorm Brand Path: {RXNORM_BRAND_PATH}")

FILE_BRAND_RXNORM_ERR = "brand_rxnorm_err.csv"
RXNORM_BRAND_ERR_PATH = OUTPUT_DIR.joinpath(FILE_BRAND_RXNORM_ERR)
print(f"RxNorm Brand Path: {RXNORM_BRAND_ERR_PATH}")

Dataset Directory: ..\..\datasets
RxNorm Brand Path: ..\..\datasets\brand_rxnorm.csv
RxNorm Brand Path: ..\..\datasets\brand_rxnorm_err.csv


In [3]:
import time
import requests
from typing import List

class RxNormConceptFetcher:
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def get_related(self, rxcui: str, ttys: List[str]):
        try:
            url = f"{self.BASE_URL}/rxcui/{rxcui}/related.json"
    
            response = requests.get(
                url,
                params={"tty": " ".join(ttys)},
                timeout=20
            )
    
            response.raise_for_status()
            data = response.json()
    
            concepts = (
                data.get("relatedGroup", {})
                .get("conceptGroup", [])
            )
    
            if not concepts:
                return None

            results = []
            for item in concepts:
                if item.get("tty") not in ttys:
                    continue
                    
                concept_properties = item.get("conceptProperties", [])
                for concept_property in concept_properties:
                    rx_cui = concept_property.get("rxcui")
                    ingredient_name = concept_property.get("name")
                    
                    results.append({
                        "rx_cui": rx_cui,
                        "name": ingredient_name,
                        "tty": concept_property.get("tty")
                    })
                
            return results
        except Exception as e:
            print(f"[ALLCONCEPTS ERROR] {tty}: {e}")
            return None

    def _get_allconcepts(self, tty: str):
        try:
            url = f"{self.BASE_URL}/allconcepts.json"
    
            response = requests.get(
                url,
                params={"tty": tty},
                timeout=20
            )
    
            response.raise_for_status()
            data = response.json()
    
            concepts = (
                data.get("minConceptGroup", {})
                .get("minConcept", [])
            )
    
            if not concepts:
                return None

            results = []
            for item in concepts:
                if item.get("tty") != tty:
                    continue
                    
                rx_cui = item.get("rxcui")
                ingredient_name = item.get("name")
                
                results.append({
                    "rx_cui": rx_cui,
                    "name": ingredient_name
                })
                
            return results
        except Exception as e:
            print(f"[ALLCONCEPTS ERROR] {tty}: {e}")
            return None

    def get_all_brand(self):
        tty = "BN"
            
        return self._get_allconcepts(tty)

In [4]:
fetcher = RxNormConceptFetcher()
brands = fetcher.get_all_brand()
print(f"Total fetched brands: {len(brands)}") 

Total fetched brands: 5120


In [6]:
from tqdm import tqdm

brand_ingredients = []
for idx, brand in tqdm(enumerate(brands), total=len(brands), desc="Processing fetch products..."):
    brand_rxcui = brand.get("rx_cui")
    brand_name = brand.get("name")
    
    products = fetcher.get_related(brand_rxcui, ["SBD"])
    for product in products:
        compositions = fetcher.get_related(product.get("rx_cui"), ["IN", "PIN"])
        ingredients = [composition for composition in compositions if composition.get("tty", "") == "IN"]
        precise_ingredients = [composition for composition in compositions if composition.get("tty") == "PIN"]

        brand_ingredients.append({
            "rx_cui": brand_rxcui,
            "name": brand_name,
            "product_rxcui": product.get("rx_cui"),
            "product_name": product.get("name"),
            "ingredients": ingredients,
            "precise_ingredients": precise_ingredients
        })
    
    

Processing fetch products...:   7%|▋         | 335/5120 [37:25<8:54:27,  6.70s/it] 


KeyboardInterrupt: 

In [7]:
import pandas as pd

df_result = pd.DataFrame(brand_ingredients)
df_result.to_csv(RXNORM_BRAND_PATH, index=False)
print(f"Brand RxNorm stored in {RXNORM_BRAND_PATH}")

df_err = pd.DataFrame(inconsistent_ing)
df_err.to_csv(RXNORM_BRAND_ERR_PATH, index=False)
print(f"Brand RxNorm Err stored in {RXNORM_BRAND_ERR_PATH}")

Brand RxNorm stored in ..\..\datasets\brand_rxnorm.csv
Brand RxNorm Err stored in ..\..\datasets\brand_rxnorm_err.csv
